# Capstone: a credit scorecard

Lenders have used the same recipe for decades. Bin each variable, replace the bins with a measure of how risky they are, fit a logistic regression on those, and turn the output into points that anyone can add up on a form. It is transparent, easy to audit and competitive with far more complex models on tabular credit data. This notebook builds one end to end using the ideas from notebooks 20 and 25.

The data are synthetic (`make_credit_data` in `common/`), because no real credit dataset downloads without an account. The source notebook used a Kaggle one (Home Credit). The method is the same, only the numbers differ.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

from common.datasets import make_credit_data

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
pd.options.display.float_format = "{:.3f}".format
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Intuition

A scorecard answers "how risky is this applicant?" with a number, built from a table of points. Age 25 to 34 might earn 12 points, utilisation above 80% might cost 30. The total maps to a risk level, and a cut-off decides who is approved.

Why bin at all? A raw variable like income has a curved, noisy relationship with default, and a straight-line model would miss it. Binning turns each variable into a few ranges, and the **weight of evidence** (WoE) of a bin says how much safer or riskier it is than average. A logistic regression on those WoE values is linear in the log-odds, which makes the points table fall out directly. Binning also handles missing values (they get their own bin) and outliers (they land in the end bin).

## 2. Maths

Count, in each bin $b$ of a variable, the goods and bads (defaulters). With $G$ and $B$ the totals,

$$\text{WoE}_b=\ln\frac{\text{share of goods in }b}{\text{share of bads in }b}=\ln\frac{g_b/G}{\text{b}_b/B}$$

A positive WoE means the bin has relatively more goods than average (safe) and a negative WoE means it is risky. **Information value** summarises a whole variable: $\text{IV}=\sum_b\bigl(\tfrac{g_b}{G}-\tfrac{b_b}{B}\bigr)\text{WoE}_b$. Rough guide: under 0.02 useless, 0.02 to 0.1 weak, 0.1 to 0.3 medium, above 0.3 strong.

**Model.** Fit a logistic regression for the good:bad odds on the WoE-coded variables:

$$\ln\text{odds}=\beta_0+\sum_j\beta_j\,\text{WoE}_j(x_j)$$

Because WoE is already a log-odds quantity, the coefficients should come out near 1 and positive. A negative one would mean two variables overlap, and is a warning sign.

**Points.** Choose a scale with three numbers: a base score at some base odds, and the **points to double the odds** (PDO). Then

$$\text{score}=\text{offset}+\text{factor}\cdot\ln\text{odds},\qquad\text{factor}=\frac{\text{PDO}}{\ln2},\quad\text{offset}=\text{base score}-\text{factor}\cdot\ln(\text{base odds})$$

Adding PDO points always doubles the good:bad odds, whatever the starting score. The total splits into one points value per bin, $\text{factor}\,(\beta_jW_{bj}+\beta_0/m)+\text{offset}/m$ for $m$ variables.

**Measures of quality.** The **Gini** coefficient is $2\,\text{AUC}-1$ (notebook 25), so 0 is useless and 1 is perfect. The **KS statistic** is the largest gap between the cumulative share of bads and the cumulative share of goods as you walk down the score. Both measure ranking power. They say nothing on calibration.

## 3. Split first, then bin on the training set only

The bin edges and WoE values are learned from the data, so they are part of the model. Fit them on the training set alone, or the test score will be optimistic (notebook 00).

In [ ]:
X, y = make_credit_data(n_samples=30000, default_rate=0.05, seed=SEED)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)
print(f"train {len(X_tr)} rows, {y_tr.sum()} bads ({y_tr.mean():.1%}); test {len(X_te)} rows")
print("missing values:", X_tr.isna().sum()[lambda s: s > 0].to_dict())

## 4. Binning and weight of evidence

Quantile bins are the simple choice: edges taken from the training data so each bin has a similar number of rows. Missing values get their own bin. I add 0.5 to each count so an empty bin does not give an infinite WoE.

In [ ]:
def fit_bins(x: pd.Series, n_bins: int = 6) -> np.ndarray:
    """Interior bin edges from training quantiles."""
    edges = np.unique(x.dropna().quantile(np.linspace(0, 1, n_bins + 1)[1:-1]).to_numpy())
    return edges


def assign_bin(x: pd.Series, edges: np.ndarray) -> pd.Series:
    idx = np.searchsorted(edges, x.to_numpy(), side="right")  # 0 .. len(edges)
    return pd.Series(np.where(x.isna(), -1, idx), index=x.index)  # -1 is the missing bin


def woe_table(x: pd.Series, y: pd.Series, edges: np.ndarray) -> pd.DataFrame:
    b = assign_bin(x, edges)
    t = pd.DataFrame({"bin": b, "bad": y}).groupby("bin")["bad"].agg(bads="sum", n="count")
    t["goods"] = t["n"] - t["bads"]
    g = (t["goods"] + 0.5) / (t["goods"].sum() + 0.5 * len(t))
    bd = (t["bads"] + 0.5) / (t["bads"].sum() + 0.5 * len(t))
    t["bad_rate"] = t["bads"] / t["n"]
    t["woe"] = np.log(g / bd)
    t["iv"] = (g - bd) * t["woe"]
    return t


edges = {c: fit_bins(X_tr[c]) for c in X_tr.columns}
tables = {c: woe_table(X_tr[c], y_tr, edges[c]) for c in X_tr.columns}
tables["utilisation"].round(3)

Utilisation shows the pattern clearly. In the lowest bins the bad rate (2.0% and 3.1%) is well under the 5% average and WoE is positive. In the highest bin it is 11%, more than double the average, and WoE is strongly negative. The relationship is monotonic, which is what a lender wants to see: more utilisation, more risk, with no unexplained reversals.

Information value ranks the variables.

In [ ]:
iv = pd.Series({c: t["iv"].sum() for c, t in tables.items()}).sort_values(ascending=False)
print(iv)
tables["employment_years"][["n", "bad_rate", "woe"]].round(3)

`delinquencies`, `utilisation` and `income` carry the most information, as they should given how the data were generated. The bin labelled -1 in `employment_years` is the missing bin. Its WoE is learned like any other, so a missing value is treated as information rather than being dropped or filled in.

## 5. The model: logistic regression on WoE

Replace each raw value by its bin's WoE, then fit a standard logistic regression. The target is "good" (1 minus default), so the log-odds are good:bad odds and the coefficients should be positive.

In [ ]:
def to_woe(frame: pd.DataFrame) -> pd.DataFrame:
    cols = {}
    for c in frame.columns:
        lookup = tables[c]["woe"]
        cols[c] = assign_bin(frame[c], edges[c]).map(lookup).fillna(0.0)  # unseen bin: neutral
    return pd.DataFrame(cols)


W_tr, W_te = to_woe(X_tr), to_woe(X_te)
good_tr = 1 - y_tr
model = LogisticRegression(C=1.0, max_iter=1000).fit(W_tr, good_tr)
print(
    pd.Series(model.coef_[0], index=W_tr.columns).round(3).to_dict(),
    " intercept:",
    model.intercept_.round(3),
)

Five coefficients are positive and close to 1, as expected from variables already in log-odds form. `loan_amount` comes out negative (-0.27), which is the warning sign from the maths section. The loan is a share of income, so once `income` is in the model `loan_amount` mostly corrects for it. A points table that penalises a safe-looking bin would be rejected at review. The standard response is to drop the variable and refit. A loan-to-income ratio would be the better feature, and is left as an exercise.

In [ ]:
used = [c for c, b in zip(W_tr.columns, model.coef_[0]) if b > 0]
W_tr, W_te = W_tr[used], W_te[used]
model = LogisticRegression(C=1.0, max_iter=1000).fit(W_tr, good_tr)
print("kept:", used)
print(
    pd.Series(model.coef_[0], index=used).round(3).to_dict(),
    " intercept:",
    model.intercept_.round(3),
)

## 6. Gini and KS on held-out data

Score the test set and compare against two other models trained on the raw data: a plain logistic regression and gradient boosting.

In [ ]:
def gini_ks(y_true: np.ndarray, risk: np.ndarray) -> tuple[float, float]:
    """Gini and KS for a risk score where higher means more likely to default."""
    fpr, tpr, _ = roc_curve(y_true, risk)
    return 2 * roc_auc_score(y_true, risk) - 1, float(np.max(tpr - fpr))


risk_card = -model.decision_function(W_te)  # higher = more likely to default
fill = X_tr.median()
raw_lr = LogisticRegression(max_iter=3000).fit((X_tr.fillna(fill) - fill) / X_tr.std(), y_tr)
risk_lr = raw_lr.decision_function((X_te.fillna(fill) - fill) / X_tr.std())
gb = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, random_state=SEED).fit(
    X_tr, y_tr
)
risk_gb = gb.predict_proba(X_te)[:, 1]

rows = [
    {"model": name, "gini": gini_ks(y_te.to_numpy(), r)[0], "ks": gini_ks(y_te.to_numpy(), r)[1]}
    for name, r in [
        ("scorecard (WoE + logistic)", risk_card),
        ("logistic on raw", risk_lr),
        ("hist gradient boosting", risk_gb),
    ]
]
pd.DataFrame(rows).set_index("model").round(3)

The scorecard (Gini 0.51) trails the plain logistic regression (0.56) and sits just under boosting (0.52). There are two reasons. Binning throws away detail inside each bin, and dropping `loan_amount` removed real signal that the raw model keeps. Giving up about 0.05 Gini buys a points table that anyone can read and audit, and that trade is the reason scorecards stay in use. A KS of 0.38 is in the normal range for consumer credit models, and a value above 0.6 usually means leakage.

## 7. Scaling to points

Use the common convention: 600 points at good:bad odds of 50:1, and 20 points to double the odds.

In [ ]:
PDO, BASE_SCORE, BASE_ODDS = 20, 600, 50
factor = PDO / np.log(2)
offset = BASE_SCORE - factor * np.log(BASE_ODDS)
m = W_tr.shape[1]
print(f"factor {factor:.2f}, offset {offset:.2f}")


def score(W: pd.DataFrame) -> np.ndarray:
    return offset + factor * model.decision_function(W)


def points_table(variable: str) -> pd.DataFrame:
    t = tables[variable].copy()
    beta = model.coef_[0][list(W_tr.columns).index(variable)]
    t["points"] = (factor * (beta * t["woe"] + model.intercept_[0] / m) + offset / m).round(0)
    return t[["n", "bad_rate", "points"]]


s_te = score(W_te)
print("score range on test:", s_te.min().round(0), "to", s_te.max().round(0))
points_table("utilisation")

The score is just a sum of per-bin points, plus nothing else, so a loan officer could compute it by hand. Check the PDO claim: take the score band by band and compare the good:bad odds.

In [ ]:
bands = pd.qcut(s_te, 8, duplicates="drop")
band_tbl = (
    pd.DataFrame({"score": s_te, "bad": y_te.to_numpy()})
    .groupby(bands, observed=True)
    .agg(mean_score=("score", "mean"), n=("bad", "count"), bad_rate=("bad", "mean"))
)
band_tbl["odds_good_bad"] = (1 - band_tbl["bad_rate"]) / band_tbl["bad_rate"]
band_tbl["log2_odds"] = np.log2(band_tbl["odds_good_bad"])
print(band_tbl.round(3))
slope = np.polyfit(band_tbl["mean_score"], band_tbl["log2_odds"], 1)[0]
print(f"points to double the odds, from the fitted slope: {1 / slope:.1f} (design: {PDO})")

The bad rate falls from 17% in the lowest band to 1% in the highest. There is one small reversal in the middle (2.0% then 2.4%), which is noise at about 25 bads per band. The observed points-to-double, measured on held-out bands, is close to the design value of 20. A lender runs this check before going live, and a large gap would mean the model is miscalibrated.

A picture of how well the score separates goods from bads, with the cut-off from notebook 25's cost logic in mind.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for label, name in [(0, "good"), (1, "bad")]:
    axes[0].hist(s_te[y_te.to_numpy() == label], bins=40, density=True, alpha=0.5, label=name)
axes[0].set(xlabel="score", ylabel="density", title="Score distributions")
axes[0].legend()
order = np.argsort(s_te)
y_sorted = y_te.to_numpy()[order]
axes[1].plot(np.cumsum(y_sorted) / y_sorted.sum(), label="bads")
axes[1].plot(np.cumsum(1 - y_sorted) / (1 - y_sorted).sum(), label="goods")
axes[1].set(
    xlabel="applicants sorted by score (low to high)",
    title="Cumulative share: KS is the biggest gap",
)
axes[1].legend()
plt.show()

The two distributions overlap heavily, and the bads sit lower. The cumulative curves show the same: by the time you have walked past the lowest-scoring 30% of applicants you have seen far more than 30% of the bads.

## 8. What breaks

- **Leakage in the binning.** Edges and WoE fitted on the full dataset leak test labels. Fit them on the training set, as above, or inside a pipeline with cross-validation.
- **Tiny bins.** A bin with 10 bads has a noisy WoE. Use fewer bins, merge neighbours, and keep the smoothing.
- **Non-monotonic bins.** If the bad rate zig-zags across ordered bins, the rule is hard to defend. Merge bins until it is monotonic, as optimal-binning libraries do.
- **Population drift.** A scorecard is built on yesterday's applicants. Compare the score distribution now and at build time with the population stability index (PSI). A PSI above 0.25 is a common trigger for a rebuild.
- **Rejected applicants.** Real lenders only see outcomes for people they approved, which biases the data. The synthetic data here has no such bias, so the problem is invisible.
- **Calibration and cost.** Gini and KS rank applicants and do not set a cut-off. The cut-off comes from costs, as in notebook 25.

The scorecard uses a logistic regression, a binning step, and a metric, all of which this section covered separately. One question is still open for the whole section: how would you tell whether more complexity (boosting, more features) is worth losing the readable points table?